# E-Commerce Analytics System
## Notebook 1: Data Generation

This notebook generates realistic synthetic datasets for:
- customers
- products
- orders
- order_items

It intentionally injects data quality issues required by the assignment:
- NULL customer IDs (~5%)
- Invalid emails (~2%)
- Wrong date formats
- Future dates
- Mixed-case product names
- Extra spaces
- Negative quantities (~3%)
- Invalid foreign keys (small %)
- Discounts >100 (edge cases)

Outputs are saved under `data/raw/`.


In [ ]:
from faker import Faker
import pandas as pd
import random
from datetime import datetime, timedelta
from pathlib import Path

fake = Faker()
random.seed(42)
Faker.seed(42)

RAW_PATH = Path("data/raw")
RAW_PATH.mkdir(parents=True, exist_ok=True)

NUM_CUSTOMERS = 500
NUM_PRODUCTS = 500
NUM_ORDERS = 500
NUM_ORDER_ITEMS = 1000


## Customer Generation

In [ ]:
def generate_customers(n):
    rows=[]
    for i in range(1,n+1):
        email=fake.email()
        if random.random()<0.02:
            email=email.replace("@","")
        rows.append({
            "customer_id":i,
            "customer_name":fake.name(),
            "email":email,
            "registration_date":fake.date_between(start_date='-3y', end_date='today'),
            "customer_type":random.choice(["REGULAR","PREMIUM","VIP"])
        })
    return pd.DataFrame(rows)

customers=generate_customers(NUM_CUSTOMERS)
customers.head()


## Product Generation

In [ ]:
categories={
"Electronics":["Mobile","Laptop","Accessories"],
"Clothing":["Men","Women","Kids"],
"Home":["Kitchen","Furniture","Decor"],
"Books":["Fiction","Education","Comics"]
}

def messy_name(name):
    if random.random()<0.15:
        return "  "+name.upper()+"  "
    return name

products=[]
for pid in range(1,NUM_PRODUCTS+1):
    cat=random.choice(list(categories.keys()))
    sub=random.choice(categories[cat])
    products.append({
        "product_id":pid,
        "product_name":messy_name(fake.word().title()+" "+fake.word().title()),
        "category":cat,
        "subcategory":sub,
        "cost_price":round(random.uniform(100,5000),2)
    })

products=pd.DataFrame(products)
products.head()


## Orders Generation

In [ ]:
statuses=["PLACED","SHIPPED","DELIVERED","CANCELLED","RETURNED"]
regions=["N","S","E","W"]

orders=[]
for oid in range(1,NUM_ORDERS+1):
    cid=random.randint(1,NUM_CUSTOMERS)
    if random.random()<0.05:
        cid=None

    dt=fake.date_time_between(start_date='-2y', end_date='now')
    date=str(dt)

    if random.random()<0.03:
        date=dt.strftime("%d-%m-%Y %H:%M:%S")

    if random.random()<0.01:
        date=str(datetime.now()+timedelta(days=90))

    orders.append({
        "order_id":oid,
        "customer_id":cid,
        "order_date":date,
        "status":random.choice(statuses),
        "region_code":random.choice(regions)
    })

orders=pd.DataFrame(orders)
orders.head()


## Order Items Generation

In [ ]:
items=[]
for iid in range(1,NUM_ORDER_ITEMS+1):
    oid=random.randint(1,NUM_ORDERS)
    if random.random()<0.01:
        oid=NUM_ORDERS+999

    qty=random.randint(1,5)
    if random.random()<0.03:
        qty=-qty
    if random.random()<0.01:
        qty=0

    disc=random.randint(0,50)
    if random.random()<0.01:
        disc=120

    items.append({
        "item_id":iid,
        "order_id":oid,
        "product_id":random.randint(1,NUM_PRODUCTS),
        "quantity":qty,
        "unit_price":round(random.uniform(150,7000),2),
        "discount_percent":disc
    })

order_items=pd.DataFrame(items)
order_items.head()


## Save CSV Files

In [ ]:
customers.to_csv(RAW_PATH/'customers.csv',index=False)
products.to_csv(RAW_PATH/'products.csv',index=False)
orders.to_csv(RAW_PATH/'orders.csv',index=False)
order_items.to_csv(RAW_PATH/'order_items.csv',index=False)

print("Files saved to", RAW_PATH.resolve())
for f in RAW_PATH.glob("*.csv"):
    print(f.name)
